# UR5e VLA bed — evaluation run (Kaggle, free)
Inputs: dataset **vla-bed-v2** and the **train notebook's output** (Add Input → Notebook output → the train notebook). Accelerator GPU T4 x2, Internet ON. Scores every checkpoint on the 100 held-out episodes, then variations and probes on the last, and selects by closed-loop success.

In [ ]:
# Full frozen-suite evaluation of a training run's checkpoints (from train.ipynb's output zip, attached as a Notebook-output input).
RUN = "baseline"
EPISODES = 100            # per suite; ~66 s per failing 100-frame episode on Kaggle's CPU → budget with MAX_HOURS
MAX_HOURS = 7.5

In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
hits = [p.parent for p in pathlib.Path("/kaggle/input").rglob("manifest.json") if (p.parent / "train").is_dir()]
assert hits, "add the private dataset vla-bed-v2 to this notebook (Add Input); found: " + str(sorted(str(p) for p in pathlib.Path("/kaggle/input").rglob("*"))[:20])
DATA = hits[0]; print("dataset root", DATA)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / "v2"; link.parent.mkdir(parents=True, exist_ok=True)
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
MEN = ROOT / "sim" / "vla-bed" / "assets" / "mujoco_menagerie"   # robot models are not vendored (BSD notices in NOTICES.md); pinned sparse clone, as scripts/pi_setup.sh does
if not (MEN / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", "https://github.com/google-deepmind/mujoco_menagerie.git", str(MEN)], check=True)
    subprocess.run(["git", "-C", str(MEN), "sparse-checkout", "set", "universal_robots_ur5e", "robotiq_2f85"], check=True)
subprocess.run(["git", "-C", str(MEN), "checkout", "--quiet", "e4049d0a3bfd58d2a3081614e6777d4007e3f86a"], check=True)
print("menagerie", subprocess.run(["git", "-C", str(MEN), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
# Find the training output zip under /kaggle/input and lay its checkpoints out where gpu/eval_all.sh expects them.
import glob, zipfile, pathlib, shutil, os
zips = glob.glob(f"/kaggle/input/**/vla-bed-{RUN}-output.zip", recursive=True)
assert zips, "attach the train notebook's output (Add Input → Notebook output) — found: " + str(glob.glob("/kaggle/input/**/*.zip", recursive=True)[:10])
tmp = pathlib.Path("/kaggle/working/unpacked"); shutil.rmtree(tmp, ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall(tmp)
dst = pathlib.Path(f"artifacts/vla-bed/{RUN}/full/checkpoints"); dst.mkdir(parents=True, exist_ok=True)
for pm in sorted(tmp.glob(f"artifacts/{RUN}/kaggle/*/pretrained_model")):
    step = pm.parent.name; (dst / step).mkdir(exist_ok=True)
    if not (dst / step / "pretrained_model").exists(): shutil.copytree(pm, dst / step / "pretrained_model")
print("checkpoints:", sorted(p.name for p in dst.iterdir()))

In [ ]:
# Every checkpoint on nominal, then variations + probes on the last, then selection by closed-loop success (R11). Wall-clock budget guarded.
import subprocess, sys, os, time, glob
t0 = time.time()
env = {**os.environ, "MUJOCO_GL": os.environ.get("MUJOCO_GL", "egl")}
cks = sorted(glob.glob(f"artifacts/vla-bed/{RUN}/full/checkpoints/*/pretrained_model"))
def ev(ck, label, *extra):
    if time.time() - t0 > MAX_HOURS * 3600: print("budget reached, skipping", label, extra); return
    r = subprocess.run([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "smolvla", "--run", RUN, "--checkpoint", ck, "--episodes", str(EPISODES), "--label", label, *extra], capture_output=True, text=True, env=env)
    print(label, extra, r.stdout[-700:], r.stderr[-400:] if r.returncode else "", f"[{(time.time()-t0)/3600:.2f} h]")
for ck in cks:
    ev(ck, f"{RUN}/{ck.split('/')[-2]}", "--variation", "nominal")
last = cks[-1]; lab = f"{RUN}/{last.split('/')[-2]}"
for var in ("camera_shift", "lighting", "target_relocation"):
    ev(last, lab, "--variation", var)
ev(last, lab, "--variation", "nominal", "--blank-image")
ev(last, lab, "--variation", "nominal", "--gain", "0.61")
print(subprocess.run([sys.executable, "sim/vla-bed/gpu/select_checkpoint.py", "--run", RUN, "--execute"], capture_output=True, text=True).stdout[-800:])

In [ ]:
import shutil, pathlib, hashlib, socket, glob
host = socket.gethostname(); stage = pathlib.Path("/kaggle/working/pack"); shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(f"sim/vla-bed/results/p5/{host}", stage / "results" / "p5" / host)
out = shutil.make_archive(f"/kaggle/working/vla-bed-{RUN}-eval", "zip", stage); shutil.rmtree(stage)
print(out, round(pathlib.Path(out).stat().st_size / 1e6, 1), "MB", "sha256", hashlib.sha256(open(out, "rb").read()).hexdigest())
print("workstation: sim/vla-bed/gpu/kaggle_import.sh <zip> <sha256>")